# Seleccion de Boltzmann — Optimizacion multiobjetivo para RuBisCO

Implementa el algoritmo de **seleccion de Boltzmann** del pipeline AI.zymes.

Concepto: seleccion probabilistica con temperatura que baja gradualmente.
- **Temperatura alta** → exploracion (muestras diversas)
- **Temperatura baja** → explotacion (mejores variantes)

**Metricas multiobjetivo:**
1. Afinidad al estado de transicion
2. Estabilidad termodinamica
3. Campo electrico en sitio activo

## 0. Imports

In [ ]:
import sys
sys.path.insert(0, '/content/analisismolecular/src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from libreria_analisismolecular import ai_zymes

sns.set_style('whitegrid')
print('Imports OK')

## 1. Generar pool inicial de variantes

In [ ]:
# Secuencia de referencia (4RUB)
REFERENCE_SEQ = 'MNTILCAISLIGDHEIGRNGDQAMKMAREAGTTVETFLTPAIPQDGLISLQSGTTTIHPYLGITPQAPTLPEEVHFLSRLLKSTRDRVIVEEYVSSPEAIVGLVTKDNGQILAALEKAGVPVNILEIVPGLVPIVMAGGTTVVEFGVFTNPLYSALLRRIAIASKDLGVPYIVSYEPVWTHGVLSAPGPGIVPDNTTVYVGGTFEDYLPKLSGHLVHVLHGRHVIDALSIIGLDNTTSSAKVGVVLSAIGVLEKVHEFGTTDGRMTKEDFIRKAGYDVPDYA'

# Generar variantes por mutacion aleatoria
np.random.seed(42)
pool = ai_zymes.generate_variant_pool(
    reference_seq=REFERENCE_SEQ,
    n_variants=100,
    mutation_rate=0.05,
)

print(f'Pool inicial: {len(pool)} variantes')
print(f'Mutaciones por variante:')
print(f'  Media: {pool["n_mutations"].mean():.1f}')
print(f'  Max: {pool["n_mutations"].max()}')
print(f'  Min: {pool["n_mutations"].min()}')
display(pool.head())

## 2. Definir funcion de scoring multiobjetivo

In [ ]:
def score_variants(population):
    """
    Calcula scores para cada variante.
    
    En produccion: calcular metricas reales con ESMFold, CASTpFold, FieldTools.
    Aca: simulacion con valores aleatorios para demostrar el algoritmo.
    """
    n = len(population)
    
    # Simular metricas (en produccion: calculos reales)
    affinity = np.random.normal(0.5, 0.1, n)       # Mayor = mejor
    stability = np.random.normal(-50.0, 5.0, n)     # Mayor (menos negativo) = mejor
    efield = np.random.normal(0.03, 0.005, n)       # Mayor = mejor
    
    # Score combinado con pesos de AI.zymes
    scores = ai_zymes.boltzmann_score(
        affinity=affinity,
        stability=stability,
        electric_field=efield,
        weights={'affinity': 0.4, 'stability': 0.3, 'electric_field': 0.3},
    )
    
    # Agregar al DataFrame
    population = population.copy()
    population['affinity'] = affinity
    population['stability'] = stability
    population['electric_field'] = efield
    population['score'] = scores
    
    return population

# Evaluar pool inicial
pool = score_variants(pool)
print(f'Scores del pool inicial:')
print(f'  Media: {pool["score"].mean():.4f}')
print(f'  Max: {pool["score"].max():.4f}')
print(f'  Min: {pool["score"].min():.4f}')

## 3. Visualizar seleccion de Boltzmann a diferentes temperaturas

In [ ]:
# Mostrar como cambia la distribucion de probabilidades con la temperatura
scores = pool['score'].values
temperatures = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, t in enumerate(temperatures):
    probs = ai_zymes.boltzmann_selection(scores, t)
    
    # Entropia de la distribucion (medida de diversidad)
    entropy = -np.sum(probs * np.log(probs + 1e-10))
    
    axes[i].bar(range(len(probs)), probs, alpha=0.7)
    axes[i].set_title(f'T = {t:.2f}\nEntropia: {entropy:.2f}')
    axes[i].set_xlabel('Variante')
    axes[i].set_ylabel('Probabilidad')

plt.tight_layout()
plt.show()

print('Observaciones:')
print('  T baja -> pocas variantes dominan (explotacion)')
print('  T alta -> distribucion uniforme (exploracion)')

## 4. Ciclo de optimizacion completo

In [ ]:
def mutate_selected(selected, reference_seq, mutation_rate=0.03):
    """Genera mutantes de las variantes seleccionadas."""
    mutants = []
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    
    for _, row in selected.iterrows():
        seq = list(row['sequence'])
        for j in range(len(seq)):
            if np.random.random() < mutation_rate:
                seq[j] = np.random.choice(list(amino_acids.replace(seq[j], '')))
        
        mutants.append({
            'id': f'mut_{len(mutants):04d}',
            'sequence': ''.join(seq),
            'mutations': 'derived',
            'n_mutations': sum(1 for a, b in zip(seq, reference_seq) if a != b),
        })
    
    return pd.DataFrame(mutants)

# Ejecutar ciclo de optimizacion
np.random.seed(42)
history = ai_zymes.boltzmann_optimization_loop(
    population=pool,
    score_fn=score_variants,
    n_iterations=50,
    t_initial=10.0,
    t_final=0.1,
    n_select=10,
    mutate_fn=lambda selected: mutate_selected(selected, REFERENCE_SEQ),
)

df_history = pd.DataFrame(history)
print(f'Optimizacion completa: {len(df_history)} iteraciones')

## 5. Visualizar convergencia

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score vs iteracion
axes[0].plot(df_history['iter'], df_history['best_score'], 'r-', label='Best score', linewidth=2)
axes[0].plot(df_history['iter'], df_history['mean_score'], 'b--', label='Mean score')
axes[0].set_xlabel('Iteracion')
axes[0].set_ylabel('Score')
axes[0].set_title('Convergencia de optimizacion')
axes[0].legend()

# Temperatura vs iteracion
axes[1].plot(df_history['iter'], df_history['temp'], 'g-', linewidth=2)
axes[1].set_xlabel('Iteracion')
axes[1].set_ylabel('Temperatura')
axes[1].set_title('Schedule de temperatura')

plt.tight_layout()
plt.show()

print(f'Mejor score final: {df_history["best_score"].iloc[-1]:.4f}')
print(f'Score inicial: {df_history["best_score"].iloc[0]:.4f}')
print(f'Mejora: {(df_history["best_score"].iloc[-1] - df_history["best_score"].iloc[0]):.4f}')

## 6. Resumen

In [ ]:
print('=== Seleccion de Boltzmann — Resumen ===')
print(f'Iteraciones: {len(df_history)}')
print(f'Temperatura: {df_history["temp"].iloc[0]:.1f} -> {df_history["temp"].iloc[-1]:.2f}')
print(f'Mejor score: {df_history["best_score"].iloc[-1]:.4f}')
print()
print('Proximo paso:')
print('- Integrar con ESMFold, CASTpFold, FieldTools en pipeline completo')
print('- Usar metricas reales en vez de simuladas')